# Temporal Graph Tutorial: Data Containers with DGF

This notebook demonstrates how to represent or load temporal graphs using DGF's
`InMemoryGraph`, including:
- **event-stream** with a synthetic example (Part 1) and a benchmark graph (Part 2), using the pipeline below
- **snapshot-series** with a job vacancy temporal graph model (Part 1.5): adjacency matrix + per-step feature matrices

## Event-stream Pipeline Overview

```
[1. Raw Data]
Event Stream: flat arrays (src, dst, timestamp, features)
        │
        ▼
[2. create_temporal_graph()]    ← Builds InMemoryGraph, GraphSchema
        │
        ▼
[3. Temporal Utilities]         ← Time slicing, chronological splitting
```

## Setup

**Custom runtime** (recommended): Build the kernel with all temporal graph
dependencies baked in — no `adhoc_import` needed:

```bash
blaze run -c opt //third_party/py/dgf:notebook_tpu
```

Then connect at `colab.corp.google.com` → Connect to local runtime →
`<hostname>.c.googlers.com:8888`.

**Or launch on XManager** (for GPU/TPU):

```bash
colab=/google/bin/releases/grp-ix-team/rapid/colab-cli/cli.par
$colab launch //third_party/py/dgf:notebook_tpu \
  --xm_resource_alloc=<your-alloc> --label="dgf_temporal"
```


In [5]:
import sys
import numpy as np

import dgf
from dgf.src.data import in_memory_graph as in_memory_graph_lib

# DGF imports (available natively in the kernel)
from dgf.src.analyse import schema as analyse_schema_lib
from google3.research.graph.graph_flow.temporal.data_loader import fetch_tgb_graph
from dgf.src.api.validate import validate_graph
from dgf.src.api.convert import graph_to_networkx

# Temporal graph utilities (direct google3 imports)
from google3.research.graph.graph_flow.temporal.utils import create_temporal_graph
from google3.research.graph.graph_flow.temporal.utils import temporal_chronological_split
from google3.research.graph.graph_flow.temporal.utils import temporal_time_slice


---

# Part 1: Synthetic Event-Stream Data

We start with a small, hand-crafted temporal graph to illustrate the core
concepts before moving to a real dataset.

### Scenario: A Social Network

5 users interact over time. Each interaction is an instantaneous event
(e.g., a message sent) with a 4-dimensional feature vector.

```
Timeline:  t=1    t=2    t=3    t=4    t=5    t=6    t=7    t=8
Events:    A→B    A→C    B→C    A→B    C→D    B→D    A→E    D→E
```

In [6]:
# ── 1a. Define raw event-stream arrays ──────────────────────────────────────

# Node IDs: A=0, B=1, C=2, D=3, E=4
NUM_NODES = 5

# 8 interaction events (src → dst at time t)
src_ids = np.array([0, 0, 1, 0, 2, 1, 0, 3], dtype=np.int64)
dst_ids = np.array([1, 2, 2, 1, 3, 3, 4, 4], dtype=np.int64)
timestamps = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=np.int64)

# Random 4-dim edge features (e.g., message embedding, sentiment, etc.)
rng = np.random.default_rng(42)
edge_features = rng.standard_normal((8, 4)).astype(np.float32)

print("Raw event stream:")
node_names = ["A", "B", "C", "D", "E"]
for i in range(len(src_ids)):
    print(
        f"  t={timestamps[i]}: {node_names[src_ids[i]]}→{node_names[dst_ids[i]]}  "
        f"feat={edge_features[i, :2].round(2)}..."
    )

Raw event stream:
  t=1: A→B  feat=[ 0.3  -1.04]...
  t=2: A→C  feat=[-1.95 -1.3 ]...
  t=3: B→C  feat=[-0.02 -0.85]...
  t=4: A→B  feat=[0.07 1.13]...
  t=5: C→D  feat=[ 0.37 -0.96]...
  t=6: B→D  feat=[-0.18 -0.68]...
  t=7: A→E  feat=[-0.43 -0.35]...
  t=8: D→E  feat=[0.41 0.43]...


### 1b. Build InMemoryGraph with `create_temporal_graph()`

`create_temporal_graph()` takes raw event-stream arrays and produces a DGF
`InMemoryGraph` where:
- **Node set**: Contains `num_nodes` entries (no node features by default).
- **Edge set**: Each edge is one interaction event, sorted by timestamp, with
  `timestamp` and `feat` features.

In [7]:
# ── 1b. Create the temporal InMemoryGraph ───────────────────────────────────

graph, schema = create_temporal_graph(
    edge_src_ids=src_ids,
    edge_dst_ids=dst_ids,
    edge_timestamps=timestamps,
    edge_features=edge_features,
    num_nodes=NUM_NODES,
)

# Validate the constructed graph against schema:
# Event stream graph stores temporal interactions in edgeset,
# while nodeset only stores num_nodes, without "#id" feature
# so warning is expected
validate_graph(graph, schema, raise_on_warning=False)
print("✓ Graph validated successfully against schema!")

# Inspect the result
print("\n=== InMemoryGraph ===")
print(f"Node sets: {list(graph.node_sets.keys())}")
print(f"  'nodes': {graph.node_sets['nodes'].num_nodes} nodes")
print(f"Edge sets: {list(graph.edge_sets.keys())}")

es = graph.edge_sets["edges"]
print(f"  'edges': {es.adjacency.shape[1]} edges")
print(f"  Edge features: {list(es.features.keys())}")
print(f"  Timestamps (sorted): {es.features['timestamp']}")
print(f"  Edge feat shape: {es.features['feat'].shape}")
print(f"  Adjacency (src): {es.adjacency[0]}")
print(f"  Adjacency (dst): {es.adjacency[1]}")

print(f"\n=== Schema ===")
dgf.print.schema(schema)

[WARNING] The nodeset 'nodes' schema is missing the '#id' feature. Giving a clearly defined #id column is recommanded. It is also required for non-string node IDs e.g. integer IDs.
✓ Graph validated successfully against schema!

=== InMemoryGraph ===
Node sets: ['nodes']
  'nodes': 5 nodes
Edge sets: ['edges']
  'edges': 8 edges
  Edge features: ['timestamp', 'feat']
  Timestamps (sorted): [1 2 3 4 5 6 7 8]
  Edge feat shape: (8, 4)
  Adjacency (src): [0 0 1 0 2 1 0 3]
  Adjacency (dst): [1 2 2 1 3 3 4 4]

=== Schema ===
Graph Schema:

Node Sets:
  nodes:
    (No features)


Edge Sets:
  edges: (Source: nodes, Target: nodes)
    | Feature   | Format     | Semantic   | Shape   | Num cat. vals   |
    |-----------|------------|------------|---------|-----------------|
    | feat      | FLOAT_32   | EMBEDDING  | (4,)    | None            |
    | timestamp | INTEGER_64 | TIMESTAMP  | None    | None            |



### 1c. Time Slicing and Chronological Splitting

The temporal utilities support efficient operations on the sorted edge list:
- **`temporal_time_slice(graph, start_time, end_time)`**: Uses binary search to
  extract events in a time window `(start_time, end_time]`. O(log E) per edge set.
- **`temporal_chronological_split(graph, val_ratio, test_ratio)`**: Splits
  into train/val/test by timestamp quantiles.

In [8]:
# ── 1c. Time slicing ────────────────────────────────────────────────────────

# Slice to events in (t=0, t=3]
# temporal_time_slice uses the schema to identify TIMESTAMP features per edge set
sliced = temporal_time_slice(graph, schema, start_time=0, end_time=3)
print("=== Time Slice (0, 3] ===")
print(f"  Edges: {sliced.edge_sets['edges'].adjacency.shape[1]}")
print(f"  Timestamps: {sliced.edge_sets['edges'].features['timestamp']}")

# Chronological split: 70% train, 15% val, 15% test
# Returns a TemporalSplit dataclass where splits are auto-inferred from the schema
split = temporal_chronological_split(
    graph, schema, val_ratio=0.15, test_ratio=0.15
)
train_g, val_g, test_g = split.train, split.val, split.test
val_time, test_time = split.val_time, split.test_time

print(f"\n=== Chronological Split ===")
print(f"  Split boundaries: val_time={val_time}, test_time={test_time}")
for name, g in [("Train", train_g), ("Val", val_g), ("Test", test_g)]:
    n_edges = g.edge_sets["edges"].adjacency.shape[1]
    ts = g.edge_sets["edges"].features["timestamp"]
    if n_edges > 0:
        print(f"  {name}: {n_edges} edges, t=[{ts[0]:.1f}, {ts[-1]:.1f}]")
    else:
        print(f"  {name}: {n_edges} edges (empty)")

=== Time Slice (0, 3] ===
  Edges: 3
  Timestamps: [1 2 3]

=== Chronological Split ===
  Split boundaries: val_time=5, test_time=6
  Train: 5 edges, t=[1.0, 5.0]
  Val: 1 edges, t=[6.0, 6.0]
  Test: 2 edges, t=[7.0, 8.0]


---

# Part 1.5: Synthetic Snapshot Graph

In many spatiotemporal settings
(traffic networks, epidemiological models, labor economics), the graph has a
**fixed topology** and time-varying **node features** — a *snapshot series*.

### Scenario: Job Vacancy Chains

We simulate a simple labor market network where **job vacancies** flow through
a fixed organizational graph over time. The model is based on the
[Vacancy Chain](https://en.wikipedia.org/wiki/Vacancy_chain) model from
organizational sociology, using an [absorbing markov chain](https://en.wikipedia.org/wiki/Absorbing_Markov_chain):

1. **Fixed topology**: An adjacency matrix `A` represents historical worker
   mobility between positions.
2. **Time-varying node features**: At each time step, each node (position)
   has a vacancy count that evolves via stochastic flows along the edges.

We'll show how this snapshot-series data can be represented as a
`dgf.InMemoryGraph` with node features as `TIMESERIES` feature type.

In [9]:
# ── 1.5a. Vacancy Chain simulator ───────────────────────────────────────────

def generate_vacancy_chain(A, steps=100, gamma=0.1, lam=0.5):
    """Generates synthetic spatiotemporal job mobility data using a Vacancy Chain model.

    The underlying mathematics track the inverse flow of workers (the movement
    of job vacancies) across a fixed network topology over discrete time steps.

    Mathematical Formulation:
    -------------------------
    1. Transition Probabilities (P):
       The probability a vacancy moves from node j to node i.
       P_{j -> i} = (1 - gamma) * [ A_{i,j} / sum_k(A_{k,j}) ]

    2. System Update (v):
       The number of vacancies at node i at time t+1.
       v_i(t+1) = sum_j( M_{j -> i}(t) ) + b_i(t)

       Where:
       - M_{j -> .}(t) ~ Multinomial( v_j(t), P_{j -> .} ) [Migrated vacancies]
       - b_i(t) ~ Poisson(lam)                             [Exogenous shocks]

    Args:
        A: ndarray (N x N). Adjacency matrix where A[i,j] is the historical
            worker flow from i to j.
        steps: Number of time steps to simulate.
        gamma: Sink probability in [0, 1]; the chance a vacancy is filled from
            outside the network.
        lam: Poisson rate for new exogenous vacancies appearing
            (company growth / retirements).

    Returns:
        X: ndarray (steps x N). Spatiotemporal feature matrix where X[t, i] is
            the vacancy count at node i at time t.
    """
    num_nodes = A.shape[0]

    # 1. Calculate the Vacancy Transition Matrix P
    # Vacancy moves j -> i based on worker moving i -> j
    in_degrees = A.sum(axis=0)
    in_degrees[in_degrees == 0] = 1  # Prevent division by zero

    P = A.T / in_degrees[:, None]    # P[j, i] = prob of vacancy moving j -> i

    # Pre-allocate the feature matrix: X[t, i] = vacancies at node i at time t
    X = np.zeros((steps, num_nodes), dtype=int)

    # Initialize day-zero vacancies
    current_vacancies = np.random.poisson(lam, size=num_nodes)

    for t in range(steps):
        next_vacancies = np.zeros(num_nodes, dtype=int)

        for j in range(num_nodes):
            if current_vacancies[j] > 0:
                # Probabilities for where node j's vacancies go
                p_internal = (1 - gamma) * P[j]

                # If node j has no incoming edges, vacancies must go to sink
                if P[j].sum() == 0:
                    p_sink = 1.0
                    p_internal = np.zeros(num_nodes)
                else:
                    p_sink = gamma

                # Build probability vector: [sink, node_0, node_1, ..., node_N]
                probs = np.insert(p_internal, 0, p_sink)

                # Sample where the vacancies move
                moves = np.random.multinomial(current_vacancies[j], probs)

                # moves[0] = vacancies that left the system (sink)
                # moves[1:] = vacancies that moved to other nodes
                next_vacancies += moves[1:]

        # Add new exogenous vacancies (retirements, company growth)
        new_vacancies = np.random.poisson(lam, size=num_nodes)
        next_vacancies += new_vacancies

        # Record the state
        X[t] = next_vacancies
        current_vacancies = next_vacancies

    return X


# ── Generate synthetic data ────────────────────────────────────────────────

np.random.seed(42)

# A small organizational mobility network (6 positions)
# A[i,j] = historical worker flow from position i to position j
A = np.array([
    [0, 3, 1, 0, 0, 0],
    [2, 0, 2, 1, 0, 0],
    [0, 1, 0, 3, 1, 0],
    [0, 0, 1, 0, 2, 1],
    [0, 0, 0, 1, 0, 2],
    [0, 0, 0, 0, 1, 0],
], dtype=float)

SNAPSHOT_STEPS = 50
SNAPSHOT_NODES = A.shape[0]

X = generate_vacancy_chain(A, steps=SNAPSHOT_STEPS, gamma=0.1, lam=0.5)

print(f"=== Vacancy Chain Simulation ===")
print(f"  Nodes (positions): {SNAPSHOT_NODES}")
print(f"  Time steps: {SNAPSHOT_STEPS}")
print(f"  Feature matrix X shape: {X.shape}")
print(f"  Total vacancies at t=0:  {X[0].sum()}")
print(f"  Total vacancies at t=49: {X[-1].sum()}")
print(f"\n  First 5 steps:\n{X[:5]}")

=== Vacancy Chain Simulation ===
  Nodes (positions): 6
  Time steps: 50
  Feature matrix X shape: (50, 6)
  Total vacancies at t=0:  3
  Total vacancies at t=49: 39

  First 5 steps:
[[3 0 0 0 0 0]
 [0 3 0 0 1 0]
 [6 0 1 1 0 0]
 [2 6 0 0 1 0]
 [5 4 2 1 1 0]]


### 1.5b. Represent as `dgf.InMemoryGraph`

A snapshot graph with fixed topology and time-varying node features can be
stored in a single `InMemoryGraph` by:

1. **Edge set**: The static adjacency `A` (one edge per non-zero entry).
2. **Node features**: The spatiotemporal matrix `X` stored as a 2D feature
   `[N, T]` on the node set, where each node has a time series of vacancy
   counts.

The accompanying `GraphSchema` declares the node feature with
`FeatureSemantic.TIMESERIES` and the edge weight with
`FeatureSemantic.NUMERICAL`, making the temporal nature of the data
explicit for downstream consumers.

In [10]:
# ── 1.5b. Build InMemoryGraph for the snapshot graph ────────────────────────

from dgf.src.data import in_memory_graph as img_lib
from dgf.src.data import schema as schema_lib

# Extract edges from the adjacency matrix (non-zero entries)
src_snap, dst_snap = np.nonzero(A)
edge_weights = A[src_snap, dst_snap].astype(np.float32)

# Static adjacency: [2, E_static]
snap_adjacency = np.stack([src_snap, dst_snap]).astype(np.int64)

# Node features: X is (T, N), transpose to (N, T) for per-node storage
node_feats_matrix = X.T.astype(np.float32)  # [N, T]

snapshot_graph = img_lib.InMemoryGraph(
    node_sets={
        'positions': img_lib.InMemoryNodeSet(
            num_nodes=SNAPSHOT_NODES,
            features={
                'vacancy_series': node_feats_matrix,  # [N, T] time series
            },
        ),
    },
    edge_sets={
        'mobility': img_lib.InMemoryEdgeSet(
            adjacency=snap_adjacency,
            features={
                'weight': edge_weights,  # Historical flow intensity
            },
        ),
    },
)

# Build schema with TIMESERIES semantic for the vacancy time series
snapshot_schema = schema_lib.GraphSchema(
    node_sets={
        'positions': schema_lib.NodeSchema(
            features={
                'vacancy_series': schema_lib.FeatureSchema(
                    format=schema_lib.FeatureFormat.FLOAT_32,
                    semantic=schema_lib.FeatureSemantic.TIMESERIES,
                    shape=(SNAPSHOT_STEPS,),
                ),
            },
        ),
    },
    edge_sets={
        'mobility': schema_lib.EdgeSchema(
            source='positions',
            target='positions',
            features={
                'weight': schema_lib.FeatureSchema(
                    format=schema_lib.FeatureFormat.FLOAT_32,
                    semantic=schema_lib.FeatureSemantic.NUMERICAL,
                ),
            },
        ),
    },
)

# Inspect the result
print("=== Snapshot InMemoryGraph ===")
print(f"Node sets: {list(snapshot_graph.node_sets.keys())}")
ns = snapshot_graph.node_sets['positions']
print(f"  'positions': {ns.num_nodes} nodes")
print(f"  Node features: {list(ns.features.keys())}")
print(f"  vacancy_series shape: {ns.features['vacancy_series'].shape}  (N × T)")

es = snapshot_graph.edge_sets['mobility']
print(f"Edge sets: {list(snapshot_graph.edge_sets.keys())}")
print(f"  'mobility': {es.adjacency.shape[1]} edges (static)")
print(f"  Edge features: {list(es.features.keys())}")
print(f"  weight: {es.features['weight'][:5]}...")

print(f"\n=== Snapshot Schema ===")
dgf.print.schema(snapshot_schema)

# Show a single snapshot at t=10
t_snapshot = 10
print(f"\n=== Snapshot at t={t_snapshot} ===")
vacancy_at_t = ns.features['vacancy_series'][:, t_snapshot]
for i in range(SNAPSHOT_NODES):
    print(f"  Position {i}: {int(vacancy_at_t[i])} vacancies")

=== Snapshot InMemoryGraph ===
Node sets: ['positions']
  'positions': 6 nodes
  Node features: ['vacancy_series']
  vacancy_series shape: (6, 50)  (N × T)
Edge sets: ['mobility']
  'mobility': 14 edges (static)
  Edge features: ['weight']
  weight: [3. 1. 2. 2. 1.]...

=== Snapshot Schema ===
Graph Schema:

Node Sets:
  positions:
    | Feature        | Format   | Semantic   | Shape   | Num cat. vals   |
    |----------------|----------|------------|---------|-----------------|
    | vacancy_series | FLOAT_32 | TIMESERIES | (50,)   | None            |


Edge Sets:
  mobility: (Source: positions, Target: positions)
    | Feature   | Format   | Semantic   | Shape   | Num cat. vals   |
    |-----------|----------|------------|---------|-----------------|
    | weight    | FLOAT_32 | NUMERICAL  | None    | None            |


=== Snapshot at t=10 ===
  Position 0: 8 vacancies
  Position 1: 10 vacancies
  Position 2: 5 vacancies
  Position 3: 3 vacancies
  Position 4: 1 vacancies
  Positio

### 1.5c. Visualize the vacancy chain dynamics

We render the vacancy chain as an animated network where:
- **Node size** scales with the vacancy count at each timestep
- **Node color** encodes vacancy intensity (darker = more vacancies)
- **Edges** show the mobility flow structure (width ∝ historical flow)

This demonstrates how a snapshot-series temporal graph evolves over time.

In [11]:
# ── 1.5c. Animate the vacancy chain ─────────────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.animation as animation
import networkx as nx
from IPython.display import HTML

# Build NetworkX MultiDiGraph using DGF's library function
G_multi = graph_to_networkx(snapshot_graph, snapshot_schema)

# Convert to DiGraph for easier visualization (since there are no multi-edges in this vacancy chain)
G = nx.DiGraph(G_multi)

# Layout using NetworkX
pos = nx.spring_layout(G, seed=42, k=2.0)

# Position labels (job titles)
position_labels = {i: f"P{i}" for i in range(SNAPSHOT_NODES)}

# Precompute global scale for consistent node sizes across frames
max_vacancy = X.max() if X.max() > 0 else 1
min_node_size = 100
max_node_size = 1500

# Edge widths from edge weights stored in the NetworkX Graph attributes
edge_weights_nx = [d['weight'] for _, _, d in G.edges(data=True)]
max_edge_w = max(edge_weights_nx) if edge_weights_nx else 1
edge_widths = [0.5 + 2.5 * w / max_edge_w for w in edge_weights_nx]

fig, ax = plt.subplots(1, 1, figsize=(8, 6))

def update(t):
    ax.clear()
    vacancies = X[t]

    # Ordered list of nodes to ensure sizes/colors correspond correctly to vacancies[i]
    nodelist = [('positions', i) for i in range(SNAPSHOT_NODES)]

    # Node sizes proportional to vacancy count
    node_sizes = [
        min_node_size + (max_node_size - min_node_size) * vacancies[i] / max_vacancy
        for i in range(SNAPSHOT_NODES)
    ]

    # Node colors: intensity encodes vacancy count
    node_colors = [vacancies[i] / max_vacancy for i in range(SNAPSHOT_NODES)]

    # Draw edges (static structure)
    nx.draw_networkx_edges(
        G, pos, ax=ax,
        width=edge_widths,
        edge_color='#cccccc',
        alpha=0.6,
        arrows=True,
        arrowsize=12,
        connectionstyle='arc3,rad=0.1',
    )

    # Draw nodes (dynamic sizes + colors)
    nx.draw_networkx_nodes(
        G, pos, ax=ax,
        nodelist=nodelist,
        node_size=node_sizes,
        node_color=node_colors,
        cmap=plt.cm.YlOrRd,
        vmin=0, vmax=1,
        edgecolors='black',
        linewidths=1.0,
    )

    # Labels: position name + vacancy count
    labels = {('positions', i): f"{position_labels[i]}\n{int(vacancies[i])}" for i in range(SNAPSHOT_NODES)}
    nx.draw_networkx_labels(G, pos, labels, ax=ax, font_size=9, font_weight='bold')

    ax.set_title(f"Vacancy Chain Dynamics  —  t = {t:3d}", fontsize=14, fontweight='bold')
    ax.set_xlim(ax.get_xlim()[0] - 0.1, ax.get_xlim()[1] + 0.1)
    ax.set_ylim(ax.get_ylim()[0] - 0.1, ax.get_ylim()[1] + 0.1)
    ax.axis('off')

anim = animation.FuncAnimation(fig, update, frames=SNAPSHOT_STEPS, interval=200, blit=False)
plt.close(fig)  # Suppress static figure; only show the animation below

# Display inline in Colab
HTML(anim.to_jshtml())

---

# Part 2: Real Data — tgbl-wiki-v2

Now we apply the same pipeline to the **tgbl-wiki-v2** dataset from the
Temporal Graph Benchmark (TGB). This dataset contains ~157k Wikipedia page
edit interactions with 172-dimensional edge features.

The dataset is hosted on CNS at:
```
/cns/iz-d/home/research-graph/public/datasets/pyg_datasets/tgb/tgbl_wiki_v2/
```

`fetch_tgb_graph()` reads the CSV directly from CNS and constructs an
`InMemoryGraph`. We then split it chronologically into train/val/test splits.

### Step 2a: Load via `fetch_tgb_graph()`

In [12]:
# ── 2a. Load tgbl-wiki-v2 via fetch_tgb_graph() ────────────────────────────

# fetch_tgb_graph() reads the CSV from CNS and builds an InMemoryGraph.
wiki_graph, wiki_schema = fetch_tgb_graph("tgbl-wiki-v2")

# Chronologically split into train/val/test splits.
wiki_split = temporal_chronological_split(wiki_graph, wiki_schema)

es = wiki_graph.edge_sets["edges"]
print(f"=== tgbl-wiki-v2 Dataset ===")
print(f"  Nodes: {wiki_graph.node_sets['nodes'].num_nodes}")
print(f"  Edges: {es.adjacency.shape[1]}")
print(f"  Edge features: {list(es.features.keys())}")
if "feat" in es.features:
    print(f"  Edge feat shape: {es.features['feat'].shape}")
print(f"  Timestamp range: [{es.features['timestamp'][0]:.0f}, {es.features['timestamp'][-1]:.0f}]")
print(f"  Split: val_time={wiki_split.val_time}, test_time={wiki_split.test_time}")
print(f"  Train edges: {wiki_split.train.edge_sets['edges'].adjacency.shape[1]}")
print(f"  Val edges:   {wiki_split.val.edge_sets['edges'].adjacency.shape[1]}")
print(f"  Test edges:  {wiki_split.test.edge_sets['edges'].adjacency.shape[1]}")

Parsing TGB edgelist: /readahead/256M/cns/iz-d/home/research-graph/public/datasets/pyg_datasets/tgb/tgbl_wiki_v2/tgbl-wiki-v2_edgelist_v2.csv
Parsed 157474 edges, 8227 unique nodes, 172 edge features
=== tgbl-wiki-v2 Dataset ===
  Nodes: 8227
  Edges: 157474
  Edge features: ['timestamp', 'feat']
  Edge feat shape: (157474, 172)
  Timestamp range: [0, 2678373]
  Split: val_time=1862645, test_time=2218288
  Train edges: 110231
  Val edges:   23622
  Test edges:  23621


### Step 2b: Inspect the training split

We now have a `TemporalSplit` with pre-split train/val/test
`InMemoryGraph` objects. Let's inspect the training split and verify the
sorted-by-timestamp invariant.

In [13]:
# ── 2b. Inspect the training split ──────────────────────────────────────────

train_graph = wiki_split.train
train_es = train_graph.edge_sets["edges"]
train_ts = train_es.features["timestamp"]

print(f"=== Training Split ===")
print(f"  Nodes: {train_graph.node_sets['nodes'].num_nodes}")
print(f"  Edges: {train_es.adjacency.shape[1]}")
print(f"  Timestamp range: [{train_ts[0]:.0f}, {train_ts[-1]:.0f}]")
print(f"  Timestamps sorted: {np.all(train_ts[:-1] <= train_ts[1:])}")
print(f"  Edge features: {list(train_es.features.keys())}")
if "feat" in train_es.features:
    print(f"  Edge feat shape: {train_es.features['feat'].shape}")

=== Training Split ===
  Nodes: 8227
  Edges: 110231
  Timestamp range: [0, 1862645]
  Timestamps sorted: True
  Edge features: ['timestamp', 'feat']
  Edge feat shape: (110231, 172)
